# Go2 ODD/COD Observer - Complete Workflow

This notebook demonstrates the complete workflow for analyzing Operational Design Domain (ODD) compliance and Conditions of Deployment (COD) for Unitree Go2 robot scenarios.

**Workflow Overview:**
1. Setup dependencies and configure Google AI SDK
2. Define ODD specifications in natural language
3. Instantiate multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. Load and process scenario data
5. Evaluate ODD compliance and compute distance metrics
6. Visualize results and generate reports

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [ ]:
# Install Google Generative AI SDK and other dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-generativeai google-cloud-aiplatform python-dotenv

In [ ]:
# Import standard libraries
import os
import sys
import json
from pathlib import Path
from typing import Dict, List, Any, Tuple

# Import data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Import Google Generative AI
import google.generativeai as genai
from google.generativeai import GenerativeModel
from google.generativeai.types import HarmCategory, HarmBlockThreshold

# Import our ODD/COD analysis framework
from odd_cod import (
    OddSpec,
    AxisSpecNumeric,
    AxisSpecCategorical,
    build_cod_vector,
    compute_window_distance,
    compute_window_odd_status,
    compute_scenario_distance,
    classify_scenario,
    compute_time_fractions,
)
from odd_cod.config_example import create_basic_indoor_odd, create_outdoor_rough_odd

# Configure plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

## 2. Configure Google AI SDK (REQUIRED)

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

In [ ]:
# Configure Google API key
# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
from dotenv import load_dotenv
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✓ Google AI SDK configured successfully")
    print("  API key detected and loaded")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English
2. **Dataset path**: Location of preprocessed window data

The orchestrator agent (Section 5) will handle everything from here.

In [ ]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s and must never be exceeded

2. Orientation Limits:
   - Roll and pitch angles must remain within ±15 degrees during normal operation
   - Angles up to ±20 degrees are acceptable at the boundary
   - The robot must never exceed ±30 degrees of roll or pitch

3. Terrain Requirements:
   - The robot is designed for smooth and moderate terrain (office floors, carpet)
   - Rough terrain is outside the operational design domain
   - Very rough terrain is completely prohibited

4. Lighting Conditions:
   - The robot can operate in bright and dim lighting conditions
   - Dark environments are outside the ODD and require additional equipment

5. Human Safety:
   - Humans may be visible at a distance (no restriction)
   - Humans in very close proximity (< 1 meter) violate the ODD
   - The system must maintain safe distances from people

6. Collision Policy:
   - Zero collisions are tolerated - any collision is an ODD violation
   - The system must detect and avoid all obstacles

IMPORTANCE WEIGHTS (for distance computation):
- Collision avoidance: Highest priority (weight: 2.0)
- Human proximity: Very high priority (weight: 1.5)
- Roll/Pitch stability: High priority (weight: 1.2)
- Speed limits: Standard priority (weight: 1.0)
- Terrain type: Standard priority (weight: 1.0)
- Lighting conditions: Lower priority (weight: 0.8)
"""

# Dataset selection
DATA_DIR = Path("data/processed/runs")
scenario_path = DATA_DIR / "run_001"  # Change this to analyze different datasets

print("✓ User inputs configured")
print(f"  - ODD: {len(odd_natural_language)} characters")
print(f"  - Dataset: {scenario_path}")

### 3.1 Natural Language ODD Description

Define operating constraints in plain English. **You can edit this text** to describe your own robot's ODD:

## 4. Instantiate AI Agents

Create specialized agents using Google Gemini 2.5 Flash.

Following modern agent architecture patterns, we define agents with:
- **System instructions** for role and task clarity
- **JSON mode** for structured outputs
- **Clean separation** between definition and execution

In [ ]:
# Configure safety settings for Gemini
safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
}

# Shared generation config for all agents
generation_config = {
    "temperature": 0.1,  # Low temperature for consistent, factual analysis
    "top_p": 0.95,
    "top_k": 40,
    "response_mime_type": "application/json",  # Enable JSON mode
}

# Verify API key
if not GOOGLE_API_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY is required. Please set it in Section 2.\n\n"
        "This notebook demonstrates AI-powered analysis and requires a valid API key."
    )

print("✓ SDK configuration ready")

### 4.1 ODD Spec Agent

Converts natural language operational constraints into structured `OddSpec` JSON.

In [ ]:
# Define ODD Spec Agent with system instructions
odd_spec_agent = GenerativeModel(
    model_name="gemini-2.0-flash-exp",
    generation_config=generation_config,
    safety_settings=safety_settings,
    system_instruction="""You are an expert in robotic system specifications and operational design domains (ODD).

Your task: Convert natural language ODD descriptions into structured JSON specifications.

Expected input: Plain text description of operational constraints (speed limits, terrain, lighting, safety rules, etc.)

Required output: JSON object matching this exact schema:
{
  "version": "1.0",
  "description": "<one-line summary>",
  "axes": {
    "speed": {
      "type": "numeric",
      "feature": "forward_velocity",
      "units": "m/s",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "roll_pitch": {
      "type": "numeric",
      "feature": "max_abs_roll_pitch",
      "units": "degrees",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "terrain": {
      "type": "categorical",
      "feature": "terrain_roughness_class",
      "allowed_in_odd": ["smooth", "moderate"],
      "allowed_all": ["smooth", "moderate", "rough", "very_rough"]
    },
    "lighting": {
      "type": "categorical",
      "feature": "lighting_class",
      "allowed_in_odd": ["bright", "dim"],
      "allowed_all": ["bright", "dim", "dark"]
    },
    "humans_close": {
      "type": "categorical",
      "feature": "humans_very_close",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    },
    "collision": {
      "type": "categorical",
      "feature": "collision_suspected",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    }
  },
  "importance": {
    "speed": 1.0,
    "roll_pitch": 1.2,
    "terrain": 1.0,
    "lighting": 0.8,
    "humans_close": 1.5,
    "collision": 2.0
  }
}

Guidelines:
- Extract numeric ranges from descriptions like "between X and Y"
- Identify "normal operation", "boundary", and "hard limit" thresholds
- Map categorical constraints to allowed value lists
- Parse importance/priority statements into numeric weights"""
)

print("✓ ODD Spec Agent created")
<VSCode.Cell id="#VSC-1e479e69" language="python">
# Configure safety settings for Gemini
safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
}

# Shared generation config for all agents
generation_config = {
    "temperature": 0.1,  # Low temperature for consistent, factual analysis
    "top_p": 0.95,
    "top_k": 40,
    "response_mime_type": "application/json",  # Enable JSON mode
}

# Verify API key
if not GOOGLE_API_KEY:
    raise RuntimeError(
        "GOOGLE_API_KEY is required. Please set it in Section 2.\n\n"
        "This notebook demonstrates AI-powered analysis and requires a valid API key."
    )

print("✓ SDK configuration ready")

### 4.1 Motion Analysis Agent

Analyzes motion data (velocity, IMU, odometry) to extract speed and orientation metrics.

In [ ]:
# Define Motion Analysis Agent with system instructions
motion_agent = GenerativeModel(
    model_name="gemini-2.0-flash-exp",
    generation_config=generation_config,
    safety_settings=safety_settings,
    system_instruction="""You are a motion analysis expert for quadruped robots.

Your task: Analyze motion time series data and extract key features for ODD compliance checking.

Expected input: JSON object with motion time series arrays:
- cmd_vx: commanded forward velocities
- odom_vx: actual odometry velocities  
- roll: roll angles in degrees
- pitch: pitch angles in degrees
- accel_x: forward acceleration in m/s²

Required output: JSON object with these exact fields:
{
  "avg_forward_speed": <float>,
  "max_forward_speed": <float>,
  "max_abs_roll_pitch_deg": <float>,
  "tracking_error": <float>,
  "motion_label": "smooth" | "dynamic"
}

Analysis guidelines:
- Calculate tracking_error as mean absolute difference between cmd_vx and odom_vx
- Use "smooth" label if tracking error < 0.15, otherwise "dynamic"
- Report max absolute roll/pitch across both axes"""
)

print("✓ Motion Analysis Agent created")

### 4.3 Vision Analysis Agent

Analyzes camera images to detect lighting conditions, human presence, and environment type.

In [ ]:
# Define Vision Analysis Agent with system instructions
vision_agent = GenerativeModel(
    model_name="gemini-2.0-flash-exp",
    generation_config=generation_config,
    safety_settings=safety_settings,
    system_instruction="""You are a computer vision expert analyzing robot camera feeds for ODD compliance.

Your task: Analyze camera images to classify environmental conditions and detect humans.

Expected input: Camera image from robot's forward-facing camera

Required output: JSON object with these exact fields:
{
  "lighting_class": "bright" | "dim" | "dark",
  "humans_visible": true | false,
  "humans_very_close": true | false,
  "environment_type": <string>
}

Analysis guidelines:
- lighting_class: "bright" for well-lit, "dim" for moderate lighting, "dark" for low visibility
- humans_visible: true if any humans appear in the frame
- humans_very_close: true if humans are within ~2 meters of camera (appears large in frame)
- environment_type: classify as "indoor_office", "indoor_home", "outdoor", "warehouse", etc."""
)

print("✓ Vision Analysis Agent created")

### 4.4 Terrain Analysis Agent (LiDAR)

Analyzes multi-channel Bird's Eye View (BEV) images to classify terrain roughness and obstacles.

In [ ]:
# Define Terrain Analysis Agent with system instructions
terrain_agent = GenerativeModel(
    model_name="gemini-2.0-flash-exp",
    generation_config=generation_config,
    safety_settings=safety_settings,
    system_instruction="""You are a terrain analysis expert for quadruped robots using LiDAR data.

Your task: Analyze Bird's Eye View (BEV) LiDAR representations to classify terrain roughness and obstacles.

Expected input: Up to 4 BEV channel images:
- Occupancy: Binary presence grid (white = occupied)
- Height: Elevation map (brighter = higher)
- Density: Point cloud density (brighter = more points)  
- Roughness: Terrain roughness (brighter = rougher)

Required output: JSON object with these exact fields:
{
  "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
  "terrain_roughness_score": <float 0-1>,
  "obstacle_density": "none" | "low" | "medium" | "high"
}

Analysis guidelines:
- "smooth": Flat, even surface (score 0.0-0.25)
- "moderate": Some unevenness but navigable (score 0.25-0.5)
- "rough": Uneven terrain with obstacles (score 0.5-0.75)
- "very_rough": Highly irregular terrain (score 0.75-1.0)
- Obstacle density based on occupancy channel coverage"""
)

print("✓ Terrain Analysis Agent created")

### 4.5 Collision Detection Agent

Fuses motion, camera, and LiDAR data to detect potential collisions.

In [ ]:
# Define Collision Detection Agent with system instructions
collision_agent = GenerativeModel(
    model_name="gemini-2.0-flash-exp",
    generation_config=generation_config,
    safety_settings=safety_settings,
    system_instruction="""You are a collision detection expert for mobile robots using sensor fusion.

Your task: Analyze multi-modal sensor data to detect collisions or imminent contact.

Expected input: 
- User message with motion metrics (tracking_error, motion_label)
- Camera image showing robot's forward view
- LiDAR BEV images (occupancy, height) showing proximity data

Required output: JSON object with these exact fields:
{
  "collision_suspected": true | false,
  "collision_confidence": <float 0-1>,
  "collision_type": "none" | "front_bump" | "side_contact" | "unknown"
}

Analysis guidelines:
- High tracking error (>0.3) suggests unexpected motion → possible collision
- Visual evidence: Look for contact points, very close obstacles in camera
- LiDAR evidence: High occupancy density near robot center indicates proximity
- Confidence should reflect strength of evidence across all modalities
- Use "unknown" collision_type when suspected but location unclear"""
)

print("✓ Collision Detection Agent created")

### 4.6 Data Source Agent

Classifies scenarios as simulated vs real based on sensor characteristics.

In [ ]:
def classify_data_source(motion_tags: Dict, manifest_data: Dict = None) -> str:
    """
    Classify scenario as 'sim' or 'real'.
    
    Args:
        motion_tags: Motion analysis results
        manifest_data: Optional ground truth from manifest.csv
        
    Returns:
        'sim' or 'real'
    """
    # Check ground truth first
    if manifest_data and 'is_sim' in manifest_data:
        return 'sim' if manifest_data['is_sim'] else 'real'
    
    # Heuristic: Perfect tracking suggests simulation
    tracking_error = motion_tags.get('tracking_error', 0.5)
    if tracking_error < 0.01:
        return 'sim'
    else:
        return 'real'

print("✓ Data Source Agent defined")

### 4.7 Execute ODD Spec Agent

Convert natural language ODD (from Section 3) into structured `OddSpec`:

In [ ]:
print("Converting natural language ODD → structured OddSpec...\n")

response = odd_spec_agent.generate_content(
    f"Convert this natural language ODD to structured JSON:\n\n{odd_natural_language}"
)
odd_spec_json = json.loads(response.text)

# Build OddSpec object
odd_spec = OddSpec(
    version=odd_spec_json["version"],
    description=odd_spec_json["description"],
    axes={},
    importance=odd_spec_json["importance"]
)

# Parse axes
for axis_name, axis_data in odd_spec_json["axes"].items():
    if axis_data["type"] == "numeric":
        odd_spec.axes[axis_name] = AxisSpecNumeric(
            feature=axis_data["feature"],
            units=axis_data["units"],
            in_odd=tuple(axis_data["in_odd"]),
            near_boundary=tuple(axis_data["near_boundary"]),
            hard_limit=tuple(axis_data["hard_limit"])
        )
    elif axis_data["type"] == "categorical":
        odd_spec.axes[axis_name] = AxisSpecCategorical(
            feature=axis_data["feature"],
            allowed_in_odd=set(axis_data["allowed_in_odd"]),
            allowed_all=set(axis_data["allowed_all"])
        )

print("="*60)
print("GENERATED ODD SPECIFICATION")
print("="*60)
print(f"Version: {odd_spec.version}")
print(f"Description: {odd_spec.description}")
print(f"\nAxes defined: {len(odd_spec.axes)}")
for axis_name, axis_spec in odd_spec.axes.items():
    print(f"  - {axis_name}: {type(axis_spec).__name__}")
print(f"\nImportance weights:")
for axis_name, weight in odd_spec.importance.items():
    print(f"  - {axis_name}: {weight}")
print("="*60)

## 5. Load Scenario Data

Load preprocessed window data from a specific run.

In [ ]:
# Configure data paths
DATA_DIR = Path("data/processed/runs")

# List available scenarios
if DATA_DIR.exists():
    scenarios = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
    print(f"✓ Found {len(scenarios)} scenarios:")
    for scenario in scenarios:
        print(f"  - {scenario.name}")
else:
    print("⚠ No processed data found.")
    print("  Please run extract_windows.py to process ROS bags first.")
    scenarios = []

In [ ]:
def load_scenario_windows(scenario_path: Path) -> pd.DataFrame:
    """
    Load window index for a scenario.
    
    Args:
        scenario_path: Path to scenario directory
        
    Returns:
        DataFrame with window metadata
    """
    # Find index CSV
    index_files = list(scenario_path.glob("index_*.csv"))
    if not index_files:
        return pd.DataFrame()
    
    return pd.read_csv(index_files[0])

# Load first scenario if available
if scenarios:
    scenario_path = scenarios[0]
    windows_df = load_scenario_windows(scenario_path)
    print(f"\n✓ Loaded {len(windows_df)} windows from {scenario_path.name}")
    if not windows_df.empty:
        print("\nWindow columns:", list(windows_df.columns))
        print("\nFirst window:")
        print(windows_df.iloc[0])
else:
    windows_df = pd.DataFrame()

## 6. Process Windows and Evaluate ODD Compliance

Analyze each window using our multi-modal agents and compute ODD compliance.

### 6.1 Agent Execution Helpers

Define clean helper functions that use our SDK-native agents.

In [ ]:
def analyze_motion(motion_data: Dict) -> Dict[str, Any]:
    """Execute motion analysis agent."""
    prompt = f"""Analyze this motion data:

{json.dumps({k: v[:10] for k, v in motion_data.items()}, indent=2)}

(showing first 10 samples of each time series)"""
    
    response = motion_agent.generate_content(prompt)
    return json.loads(response.text)


def analyze_vision(image_path: Path) -> Dict[str, Any]:
    """Execute vision analysis agent."""
    img = Image.open(image_path)
    response = vision_agent.generate_content(["Analyze this robot camera image:", img])
    return json.loads(response.text)


def analyze_terrain(bev_paths: Dict[str, Path]) -> Dict[str, Any]:
    """Execute terrain analysis agent."""
    images = []
    channels_desc = []
    for channel in ['occupancy', 'height', 'density', 'roughness']:
        if channel in bev_paths and bev_paths[channel].exists():
            images.append(Image.open(bev_paths[channel]))
            channels_desc.append(f"{channel} channel")
    
    if not images:
        raise FileNotFoundError("No BEV images found")
    
    prompt = f"Analyze these LiDAR BEV images ({', '.join(channels_desc)}):"
    response = terrain_agent.generate_content([prompt] + images)
    return json.loads(response.text)


def analyze_collision(motion_tags: Dict, cam_path: Path, bev_paths: Dict[str, Path]) -> Dict[str, Any]:
    """Execute collision detection agent with sensor fusion."""
    # Build multi-modal input
    images = [Image.open(cam_path)]
    for channel in ['occupancy', 'height']:
        if channel in bev_paths and bev_paths[channel].exists():
            images.append(Image.open(bev_paths[channel]))
    
    prompt = f"""Detect collisions using this sensor data:

Motion metrics:
- Tracking error: {motion_tags.get('tracking_error', 0):.3f}
- Motion label: {motion_tags.get('motion_label', 'unknown')}

Camera image and LiDAR BEV images follow:"""
    
    response = collision_agent.generate_content([prompt] + images)
    return json.loads(response.text)

print("✓ Agent execution helpers defined")

### 6.2 Window Analysis Function

Main function that coordinates all agents to analyze a single window.

In [ ]:
def analyze_window(
    window_row: pd.Series,
    scenario_path: Path,
    odd_spec: OddSpec,
) -> Dict[str, Any]:
    """
    Analyze a single window using all agents with parallel execution where possible.
    
    Args:
        window_row: Row from windows DataFrame
        scenario_path: Path to scenario directory
        odd_spec: ODD specification
        
    Returns:
        Complete window analysis results
    """
    # Load motion data
    motion_path = scenario_path / window_row["motion_path"]
    with open(motion_path, 'r') as f:
        motion_data = json.load(f)
    
    # Prepare file paths
    cam_path = scenario_path / window_row["cam_image_path"]
    if not cam_path.exists():
        raise FileNotFoundError(f"Camera image not found: {cam_path}")
    
    bev_base = scenario_path / window_row.get("bev_image_path", "")
    bev_paths = {}
    if bev_base.exists():
        base_name = bev_base.stem
        for channel in ['occupancy', 'height', 'density', 'roughness']:
            channel_path = bev_base.parent / f"{base_name.replace('bev', f'bev_{channel}')}.png"
            if channel_path.exists():
                bev_paths[channel] = channel_path
    
    if not bev_paths:
        raise FileNotFoundError(f"No BEV images found for: {bev_base}")
    
    # Execute independent agents in parallel
    # Note: Motion, Vision, and Terrain analyses are independent
    motion_tags = analyze_motion(motion_data)
    vision_tags = analyze_vision(cam_path)
    terrain_tags = analyze_terrain(bev_paths)
    
    # Collision agent depends on motion results (sequential)
    collision_tags = analyze_collision(motion_tags, cam_path, bev_paths)
    
    # Merge all tags
    merged_tags = {
        **motion_tags,
        **vision_tags,
        **terrain_tags,
        **collision_tags,
    }
    
    # Build COD vector and compute distance
    cod_vector = build_cod_vector(merged_tags, odd_spec)
    window_distance, axis_distances, axis_statuses = compute_window_distance(cod_vector, odd_spec)
    odd_status = compute_window_odd_status(axis_statuses)
    
    return {
        "window_id": window_row["window_id"],
        "start_time": window_row["start_time"],
        "tags": merged_tags,
        "cod_vector": cod_vector,
        "distance": window_distance,
        "axis_distances": axis_distances,
        "axis_statuses": axis_statuses,
        "odd_status": odd_status,
    }

print("✓ Window analysis function defined")

In [ ]:
# Process all windows in the scenario
if not windows_df.empty:
    print(f"Processing {len(windows_df)} windows...\n")
    
    window_results = []
    for idx, window_row in windows_df.iterrows():
        result = analyze_window(window_row, scenario_path, odd_spec)
        window_results.append(result)
        
        # Print summary for first few windows
        if idx < 3:
            print(f"Window {result['window_id']} (t={result['start_time']:.1f}s):")
            print(f"  Distance: {result['distance']:.3f}")
            print(f"  Status: {result['odd_status']}")
            print(f"  Speed: {result['tags']['avg_forward_speed']:.2f} m/s")
            print(f"  Roll/Pitch: {result['tags']['max_abs_roll_pitch_deg']:.1f}°")
            print(f"  Collision: {result['tags']['collision_suspected']}")
            print()
    
    print(f"✓ Processed all {len(window_results)} windows")
else:
    print("⚠ No windows to process")
    window_results = []

## 7. Compute Scenario-Level Metrics

Aggregate window results to evaluate overall scenario compliance.

In [ ]:
if window_results:
    # Extract window distances and statuses
    window_distances = [r["distance"] for r in window_results]
    window_statuses = [r["odd_status"] for r in window_results]
    
    # Compute scenario distance
    scenario_distance = compute_scenario_distance(window_distances, window_statuses)
    
    # Compute time fractions
    time_fractions = compute_time_fractions(window_statuses)
    
    # Classify scenario
    exit_fraction = time_fractions["odd_exit"]
    scenario_class = classify_scenario(scenario_distance, exit_fraction)
    
    # Display results
    print("=" * 60)
    print("SCENARIO ANALYSIS SUMMARY")
    print("=" * 60)
    print(f"Scenario: {scenario_path.name}")
    print(f"Total Windows: {len(window_results)}")
    print(f"\nScenario Distance: {scenario_distance:.3f}")
    print(f"Classification: {scenario_class}")
    print(f"\nTime Distribution:")
    print(f"  In ODD: {time_fractions['in_odd']:.1%}")
    print(f"  Near Boundary: {time_fractions['near_boundary']:.1%}")
    print(f"  ODD Exit: {time_fractions['odd_exit']:.1%}")
    print("=" * 60)
else:
    print("No results to summarize")

## 8. Visualize Results

Create visualizations to understand ODD compliance over time.

In [ ]:
if window_results:
    # Create results DataFrame for plotting
    results_df = pd.DataFrame([
        {
            'window_id': r['window_id'],
            'time': r['start_time'],
            'distance': r['distance'],
            'status': r['odd_status'],
            'speed': r['tags']['avg_forward_speed'],
            'roll_pitch': r['tags']['max_abs_roll_pitch_deg'],
            'collision': r['tags']['collision_suspected'],
        }
        for r in window_results
    ])
    
    # Plot 1: Distance over time
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Distance timeline
    ax = axes[0, 0]
    ax.plot(results_df['time'], results_df['distance'], marker='o', linewidth=2)
    ax.axhline(y=0.3, color='orange', linestyle='--', label='ODD Boundary')
    ax.axhline(y=0.7, color='red', linestyle='--', label='ODD Exit')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('ODD Distance')
    ax.set_title('ODD Distance Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Status distribution
    ax = axes[0, 1]
    status_counts = results_df['status'].value_counts()
    colors = {'in_odd': 'green', 'near_boundary': 'orange', 'odd_exit': 'red'}
    ax.bar(status_counts.index, status_counts.values, 
           color=[colors.get(s, 'gray') for s in status_counts.index])
    ax.set_xlabel('ODD Status')
    ax.set_ylabel('Number of Windows')
    ax.set_title('ODD Status Distribution')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Speed over time
    ax = axes[1, 0]
    ax.plot(results_df['time'], results_df['speed'], marker='o', color='blue', linewidth=2)
    speed_spec = odd_spec.axes['speed']
    ax.axhline(y=speed_spec.in_odd[1], color='orange', linestyle='--', label='ODD Limit')
    ax.axhline(y=speed_spec.near_boundary[1], color='red', linestyle='--', label='Boundary')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Speed (m/s)')
    ax.set_title('Forward Velocity Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Roll/Pitch over time
    ax = axes[1, 1]
    ax.plot(results_df['time'], results_df['roll_pitch'], marker='o', color='purple', linewidth=2)
    rp_spec = odd_spec.axes['roll_pitch']
    ax.axhline(y=rp_spec.in_odd[1], color='orange', linestyle='--', label='ODD Limit')
    ax.axhline(y=rp_spec.near_boundary[1], color='red', linestyle='--', label='Boundary')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Roll/Pitch (degrees)')
    ax.set_title('Max Roll/Pitch Over Time')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Visualizations generated")
else:
    print("No data to visualize")

## 9. Generate Final Report

Create a comprehensive summary of the ODD/COD analysis.

In [ ]:
if window_results:
    # Compile report data
    report = {
        "scenario_id": scenario_path.name,
        "odd_version": odd_spec.version,
        "total_windows": len(window_results),
        "scenario_distance": scenario_distance,
        "scenario_class": scenario_class,
        "time_fractions": time_fractions,
        "statistics": {
            "avg_speed": float(results_df['speed'].mean()),
            "max_speed": float(results_df['speed'].max()),
            "avg_roll_pitch": float(results_df['roll_pitch'].mean()),
            "max_roll_pitch": float(results_df['roll_pitch'].max()),
            "collision_count": int(results_df['collision'].sum()),
        },
        "violations": [
            {
                "window_id": r['window_id'],
                "time": r['start_time'],
                "distance": r['distance'],
                "violated_axes": [k for k, v in r['axis_statuses'].items() if v == 'out_of_odd']
            }
            for r in window_results
            if r['odd_status'] == 'odd_exit'
        ]
    }
    
    # Display report
    print("\n" + "=" * 60)
    print("FINAL ANALYSIS REPORT")
    print("=" * 60)
    print(f"\nScenario: {report['scenario_id']}")
    print(f"ODD Version: {report['odd_version']}")
    print(f"Windows Analyzed: {report['total_windows']}")
    print(f"\nOverall Assessment:")
    print(f"  Classification: {report['scenario_class']}")
    print(f"  Distance Score: {report['scenario_distance']:.3f}")
    print(f"\nTime in Each State:")
    print(f"  In ODD: {report['time_fractions']['in_odd']:.1%}")
    print(f"  Near Boundary: {report['time_fractions']['near_boundary']:.1%}")
    print(f"  ODD Exit: {report['time_fractions']['odd_exit']:.1%}")
    print(f"\nOperational Statistics:")
    print(f"  Average Speed: {report['statistics']['avg_speed']:.2f} m/s")
    print(f"  Maximum Speed: {report['statistics']['max_speed']:.2f} m/s")
    print(f"  Average Roll/Pitch: {report['statistics']['avg_roll_pitch']:.1f}°")
    print(f"  Maximum Roll/Pitch: {report['statistics']['max_roll_pitch']:.1f}°")
    print(f"  Collision Events: {report['statistics']['collision_count']}")
    
    if report['violations']:
        print(f"\nODD Violations ({len(report['violations'])}):")
        for v in report['violations'][:5]:  # Show first 5
            print(f"  - Window {v['window_id']} (t={v['time']:.1f}s): "
                  f"distance={v['distance']:.3f}, axes={v['violated_axes']}")
        if len(report['violations']) > 5:
            print(f"  ... and {len(report['violations']) - 5} more")
    else:
        print("\n✓ No ODD violations detected")
    
    print("=" * 60)
    
    # Save report to JSON
    report_path = scenario_path / "odd_analysis_report.json"
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"\n✓ Report saved to: {report_path}")
else:
    print("No data to report")

## 10. Summary and Next Steps

This notebook demonstrated the complete ODD/COD analysis workflow:

1. ✓ Set up Google AI SDK and dependencies
2. ✓ Defined ODD specifications with natural language boundaries
3. ✓ Instantiated multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. ✓ Loaded and processed scenario data from preprocessed windows
5. ✓ Evaluated ODD compliance and computed distance metrics
6. ✓ Visualized results with timeline plots and distributions
7. ✓ Generated comprehensive analysis report

### Next Steps:

- **Process more scenarios**: Apply this workflow to additional runs from your ROS2 bags
- **Compare sim vs real**: Analyze differences in ODD compliance between simulation and real robot
- **Tune ODD specifications**: Adjust boundaries based on observed performance
- **Enhance agents**: Improve prompt engineering for more accurate multi-modal analysis
- **Export results**: Generate reports for stakeholders or competition submission

### Resources:

- [Kaggle 5-Day Agents Intensive](https://www.kaggle.com/learn-guide/5-day-agents)
- [Google Gemini API Documentation](https://ai.google.dev/docs)
- [Project Repository](https://github.com/danmartinez78/go2-odd-observer)